# OneVoice V2 — Qualcomm hosted-device profile
Clone source from GitHub. Read the frozen model from Drive and write hosted-device evidence back to Drive. This is not field-device validation.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
DRIVE_ROOT = Path('/content/drive/MyDrive/OneVoice')
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'onnxruntime', 'qai-hub'], check=True)
token = userdata.get('QAI_HUB_API_TOKEN')
if not token: raise RuntimeError('Add QAI_HUB_API_TOKEN to Colab Secrets and enable notebook access')
os.environ['QAI_HUB_API_TOKEN'] = token
MODEL = DRIVE_ROOT / 'models/frozen/model.onnx'
INPUTS = DRIVE_ROOT / 'models/frozen/correctness_input.npz'
REPORT_DIR = DRIVE_ROOT / 'reports/qai/frozen'
DEVICE = '<exact Snapdragon 8 Gen 3 device name from AI Hub>'
print('Source:', REPO, '| Model:', MODEL, '| Reports:', REPORT_DIR)


In [ ]:
if DEVICE.startswith('<'):
    raise ValueError('Set DEVICE to an exact Qualcomm AI Hub hosted-device name')
subprocess.run([sys.executable, 'notebooks/export_qai.py', '--model', str(MODEL), '--inputs', str(INPUTS), '--device', DEVICE, '--report-dir', str(REPORT_DIR)], check=True)


In [ ]:
import json
json.loads((REPORT_DIR / 'qai_report.json').read_text(encoding='utf-8'))
